In [1]:
import queue
import copy
from random import choice
from copy import deepcopy
import heapq as h

class Node:
    def __init__(self, state, parent=None, action=None, cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost
        self.depth = 0 if parent is None else parent.depth + 1
    def __hash__(self):
        return hash(str(self.state))
    def __eq__(self, other):
        if type(self.state) == 'list':
            sameAll = True
            for i in range(0,3):
                sameRow = True
                for j in range(0,3):
                    if self.state[i][j] != other[i][j]:
                        sameRow = False
                if sameRow == False:
                    sameAll = False
                    break
            return sameAll
        else:
            return True if self.state == other else False

In [2]:
class TravelPlanProblem:
    def __init__(self, initial_state, goal_test, state_transition_model, path_cost=0, actions=""):
        self.state = initial_state
        self.goal_state = goal_test
        self.state_transition_model = state_transition_model
        self.path_cost = path_cost
        self.actions = actions
    def is_goal(self, current_state):
        return self.goal_state == current_state
    def get_valid_actions(self, state):
        return self.state_transition_model[state].keys()
    def apply_action(self, state, action):
        return action
    def expand_node(self, node):
        nodes = []
        for action in self.get_valid_actions(node.state):
            chnode = Node(self.apply_action(node.state,action),parent = node,action=action,cost= node.cost + self.state_transition_model[node.state][action])
            nodes.append(chnode)
        return nodes

    def printNode(self, message, node):
        print("Action =", node.action, message, node.state)

In [3]:
class GeneralSearch:
    def __init__(self, problem, search_strategy="breadth_first"):
        self.problem = problem
        self.frontier = []
        h.heappush(self.frontier,(self.problem.path_cost,self.problem.state,Node(self.problem.state)))
    def search(self, max_depth=None):
        visited = set()
        while(self.frontier):
            current_pos = h.heappop(self.frontier)
            if current_pos[2].state in visited:
                continue
            if current_pos[2].state == self.problem.goal_state:
                return current_pos
            ss = self.problem.expand_node(current_pos[2])
            for n in ss:
                h.heappush(self.frontier,(n.cost,n.state,n))
            visited.add(current_pos[1])
            current_pos = h.heappop(self.frontier)
        return None

In [8]:
state_transition_model = {
    'Adrar': {'Tindouf': 610, 'Bechar': 540, 'El Bayadh': 790, 'Laghouat': 840, 'Ghardaia': 690, 'Tamanrasset': 1280},
    'Chlef': {'Mostaganem': 110, 'Relizane': 95, 'Biskra': 340, 'Tissemsilt': 70, 'Ain Defla': 55, 'Tipaza': 85},
    'Laghouat': {'El Bayadh': 220, 'Ghardaia': 250, 'Djelfa': 160, 'Tiaret': 240},
    'Oum El Bouaghi': {'Batna': 75, 'Khenchela': 90, 'Tbessa': 200, 'Souk Ahras': 180, 'Guelma': 130, 'Constantine': 110, 'Mila': 95},
    'Batna': {"MSila": 160, 'Biskra': 120, 'Khenchela': 80, 'Oum El Bouaghi': 75, 'Mila': 140, 'Setif': 130},
    'Bejaia': {'Tizi Ouzou': 60, 'Bouira': 110, 'Bordj Bou Arreridj': 140, 'Setif': 90, 'Jijel': 70},
    'Biskra': {'Djelfa': 180, "MSila": 150, 'Batna': 120, 'Khenchela': 160, 'El Oued': 210, 'Ouargla': 280},
    'Bechar': {'Naama': 320, 'El Bayadh': 470, 'Adrar': 540, 'Tindouf': 610},
    'Blida': {'Tipaza': 45, 'Algiers': 35, 'Bouira': 65, 'Medea': 55, 'Ain Defla': 40},
    'Bouira': {'Medea': 80, 'Blida': 65, 'Boumerdes': 70, 'Tizi Ouzou': 50, 'Bordj Bou Arreridj': 90, "MSila": 110},
    'Tamanrasset': {'Adrar': 1280, 'Ghardaia': 960, 'Ouargla': 820, 'Illizi': 950},
    'Tbessa': {'El Oued': 250, 'Khenchela': 200, 'Oum El Bouaghi': 180, 'Souk Ahras': 150},
    'Tlemcen': {'Ain Temouchent': 70, 'Sidi Bel Abbes': 90, 'Naama': 160},
    'Tiaret': {'El Bayadh': 240, 'Saefda': 180, 'Mascara': 140, 'Relizane': 120, 'Tissemsilt': 90, 'Djelfa': 160, 'Laghouat': 240},
    'Tizi Ouzou': {'Bouira': 50, 'Boumerdes': 70, 'Bejaia': 60},
    'Algiers': {'Blida': 35, 'Tipaza': 45, 'Boumerdes': 30},
    'Djelfa': {'Laghouat': 160, 'Tiaret': 160, 'Tissemsilt': 130, 'Medea': 110, "MSila": 120, 'Biskra': 180, 'Ouargla': 320, 'Ghardaia': 250},
    'Jijel': {'Mila': 110, 'Setif': 130, 'Bejaia': 70, 'Skikda': 90},
    'Setif': {'Batna': 130, "MSila": 90, 'Bordj Bou Arreridj': 70, 'Bejaia': 90, 'Jijel': 130, 'Mila': 80},
    'Saefda': {'El Bayadh': 180, 'Sidi Bel Abbes': 100, 'Mascara': 80, 'Tizi Ouzou': 160},
    'Skikda': {'Constantine': 80, 'Jijel': 90, 'Annaba': 110, 'Guelma': 140},
    'Sidi Bel Abbes': {'Naama': 160, 'Tlemcen': 90, 'Ain Temouchent': 70, 'Oran': 60, 'Mascara': 80, 'Saefda': 100, 'El Bayadh': 240},
    'Annaba': {'Guelma': 110, 'Skikda': 110, 'El Tarf': 60},
    'Guelma': {'Oum El Bouaghi': 130, 'Constantine': 110, 'Skikda': 140, 'Annaba': 110, 'El Tarf': 90, 'Souk Ahras': 120},
    'Constantine': {'Oum El Bouaghi': 110, 'Mila': 80, 'Jijel': 110, 'Skikda': 80, 'Guelma': 110},
    'Medea': {'Djelfa': 110, 'Tissemsilt': 90, 'Ain Defla': 40, 'Blida': 55, 'Bouira': 80, "MSila": 70},
    'Mostaganem': {'Oran': 60, 'Mascara': 50, 'Relizane': 40, 'Chlef': 110},
    "MSila": {'Djelfa': 120, 'Medea': 70, 'Bouira': 110, 'Bordj Bou Arreridj': 70, 'Setif': 90, 'Batna': 160, 'Biskra': 150},
    'Mascara': {'Saefda': 80, 'Sidi Bel Abbes': 80, 'Oran': 50, 'Mostaganem': 50, 'Relizane': 60, 'Tiaret': 140},
    'Ouargla': {'Ghardaia': 320, 'El Oued': 280, 'Illizi': 680, 'Tamanrasset': 820},
    'Oran': {'Ain Temouchent': 70, 'Sidi Bel Abbes': 60, 'Mascara': 50, 'Mostaganem': 60},
    'El Bayadh': {'Adrar': 790, 'Bechar': 470, 'Naama': 210, 'Sidi Bel Abbes': 240, 'Saefda': 180, 'Tiaret': 240, 'Laghouat': 220, 'Ghardaia': 250},
    'Illizi': {'Tamanrasset': 950, 'Ouargla': 680},
    'Bordj Bou Arreridj': {"MSila": 70, 'Bouira': 90, 'Bejaia': 140, 'Setif': 70},
    'Boumerdes': {'Blida': 30, 'Algiers': 30, 'Tizi Ouzou': 70, 'Bouira': 70},
    'El Tarf': {'Souk Ahras': 90, 'Guelma': 90, 'Annaba': 60},
    'Tindouf': {'Bechar': 610, 'Adrar': 610},
    'Tissemsilt': {'Relizane': 40, 'Chlef': 70, 'Ain Defla': 90, 'Medea': 90, 'Djelfa': 130, 'Tiaret': 90},
    'El Oued': {'Ouargla': 280, 'Biskra': 210, 'Khenchela': 250, 'Tbessa': 250},
    'Khenchela': {'El Oued': 250, 'Biskra': 160, 'Batna': 80, 'Oum El Bouaghi': 90, 'Tbessa': 200},
    'Souk Ahras': {'Tbessa': 150, 'Oum El Bouaghi': 180, 'Guelma': 120, 'El Tarf': 90},
    'Tipaza': {'Ain Defla': 40, 'Chlef': 85, 'Algiers': 45, 'Blida': 45},
    'Mila': {'Batna': 140, 'Setif': 80, 'Jijel': 110, 'Constantine': 80, 'Oum El Bouaghi': 95},
    'Ain Defla': {'Tissemsilt': 90, 'Chlef': 55, 'Tipaza': 40, 'Blida': 40, 'Medea': 40},
    'Naama': {'Tlemcen': 160, 'Sidi Bel Abbes': 160, 'El Bayadh': 210, 'Bechar': 320},
    'Ain Temouchent': {'Tlemcen': 70, 'Sidi Bel Abbes': 70, 'Oran': 70},
    'Ghardaia': {'Adrar': 690, 'El Bayadh': 250, 'Laghouat': 250, 'Djelfa': 250, 'Ouargla': 320, 'Tamanrasset': 960},
    'Relizane': {'Mascara': 60, 'Mostaganem': 40, 'Chlef': 95, 'Tissemsilt': 40, 'Tiaret': 120}
}
initial_city = "Tlemcen"
goal_city = "Illizi"

travel_plan_problem = TravelPlanProblem(initial_city, goal_city, state_transition_model)
copy_travel_plan = deepcopy(travel_plan_problem)

In [9]:
def get_solution_path(solution_node):
    path = []
    while solution_node:
        path.insert(0, solution_node.state)
        solution_node = solution_node.parent
    return path

solution_node_breadth_first = GeneralSearch(travel_plan_problem, search_strategy="breadth_first").search()

if solution_node_breadth_first != False:
    sl = solution_node_breadth_first[2]
    print("-" * 75)
    print(" BFS — Travel Planning ".center(75, "-"))
    print("-" * 75)
    print("Solution Path (BFS):", get_solution_path(sl))
    print("Total Cost    (BFS):", sl.cost, "km")

---------------------------------------------------------------------------
-------------------------- BFS — Travel Planning --------------------------
---------------------------------------------------------------------------
Solution Path (BFS): ['Tlemcen', 'Sidi Bel Abbes', 'Mascara', 'Tiaret', 'Djelfa', 'Ouargla', 'Illizi']
Total Cost    (BFS): 1470 km
